1. Importar bibliotecas e configurar caminhos

In [2]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

In [3]:
RANDOM_STATE = 42
PROJECT_DIR = Path.cwd().parent

MODEL_DIR = PROJECT_DIR / "models"
DATA_DIR = PROJECT_DIR / "data" / "raw"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "mlp_mnist.joblib"

print(f"Arquivo do modelo: {MODEL_PATH.resolve()}")
print(f"Cache dos dados:   {DATA_DIR.resolve()}")

Arquivo do modelo: D:\Users\Deepen\Box\Box Sync\Capacitação\Cursos Online\SENAI\Desenvolvimento IA M2\Projeto final\models\mlp_mnist.joblib
Cache dos dados:   D:\Users\Deepen\Box\Box Sync\Capacitação\Cursos Online\SENAI\Desenvolvimento IA M2\Projeto final\data\raw


2. Carregar a base MNIST

In [5]:
inicio = perf_counter()

mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False,
    parser="auto",
    data_home=str(DATA_DIR),
)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(np.uint8)

print(f"Dimensão de X: {X.shape}")
print(f"Dimensão de y: {y.shape}")
print(f"Tempo de carregamento: {perf_counter() - inicio:.2f} s")

Dimensão de X: (70000, 784)
Dimensão de y: (70000,)
Tempo de carregamento: 146.46 s


3. Separar treino, validação e teste

In [6]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.125,
    random_state=RANDOM_STATE,
    stratify=y_dev,
)

# Após definir a arquitetura, utilizamos todo o conjunto de desenvolvimento.
X_fit = np.concatenate((X_train, X_val), axis=0)
y_fit = np.concatenate((y_train, y_val), axis=0)

print(f"Treino original:       {len(X_train):,}")
print(f"Validação:             {len(X_val):,}")
print(f"Treino final (70%+10%): {len(X_fit):,}")

Treino original:       49,000
Validação:             7,000
Treino final (70%+10%): 56,000


4. Pré-processar os pixels

In [7]:
X_fit = X_fit / 255.0
X_test = X_test / 255.0

print(f"Formato de entrada: {X_fit.shape[1]} atributos")
print(f"Faixa dos pixels: {X_fit.min():.1f} a {X_fit.max():.1f}")

Formato de entrada: 784 atributos
Faixa dos pixels: 0.0 a 1.0


5. Treinar o MLP selecionado (usando os parâmetros vencedores do trabalho anterior, pois o foco aqui não será testar vários modelos, mas usar o melhor modelo via API)

In [8]:
mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation="relu",
    max_iter=200,
    random_state=RANDOM_STATE,
)

inicio = perf_counter()
mlp.fit(X_fit, y_fit)
tempo_treino = perf_counter() - inicio

print(f"Iterações realizadas: {mlp.n_iter_}")
print(f"Tempo de treinamento: {tempo_treino:.2f} s")
print(f"Convergência antes do limite: {mlp.n_iter_ < mlp.max_iter}")

Iterações realizadas: 57
Tempo de treinamento: 37.61 s
Convergência antes do limite: True


6. Avaliar no conjunto de teste

In [10]:
inicio = perf_counter()
y_pred = mlp.predict(X_test)
tempo_predicao = perf_counter() - inicio

acuracia = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Acurácia no teste:       {acuracia:.4%}")
print(f"F1-score macro no teste: {f1_macro:.4%}")
print(f"Tempo de predição:       {tempo_predicao:.2f} s")
print()
print(classification_report(y_test, y_pred, digits=4))

Acurácia no teste:       97.9571%
F1-score macro no teste: 97.9394%
Tempo de predição:       0.04 s

              precision    recall  f1-score   support

           0     0.9807    0.9935    0.9871      1381
           1     0.9855    0.9917    0.9886      1575
           2     0.9841    0.9735    0.9788      1398
           3     0.9797    0.9790    0.9793      1428
           4     0.9815    0.9692    0.9753      1365
           5     0.9769    0.9707    0.9738      1263
           6     0.9841    0.9884    0.9862      1375
           7     0.9748    0.9822    0.9785      1459
           8     0.9822    0.9714    0.9768      1365
           9     0.9658    0.9734    0.9696      1391

    accuracy                         0.9796     14000
   macro avg     0.9795    0.9793    0.9794     14000
weighted avg     0.9796    0.9796    0.9796     14000



In [11]:
print("Dimensão de X_test:", X_test.shape)
print("Quantidade de previsões:", len(y_pred))
print(f"Tempo de predição: {tempo_predicao:.2f} s")

Dimensão de X_test: (14000, 784)
Quantidade de previsões: 14000
Tempo de predição: 0.04 s


7. Serializar o modelo (para podermos usar, sem precisar retreinar)

In [14]:
joblib.dump(mlp, MODEL_PATH)

tamanho_mb = MODEL_PATH.stat().st_size / (1024 ** 2)

print(f"Modelo salvo em: {MODEL_PATH.resolve()}")
print(f"Tamanho do arquivo: {tamanho_mb:.2f} MB")


Modelo salvo em: D:\Users\Deepen\Box\Box Sync\Capacitação\Cursos Online\SENAI\Desenvolvimento IA M2\Projeto final\models\mlp_mnist.joblib
Tamanho do arquivo: 1.22 MB


8. Conferir o artefato salvo

In [13]:
mlp_carregado = joblib.load(MODEL_PATH)
amostras_teste = X_test[:5]

predicoes_originais = mlp.predict(amostras_teste)
predicoes_recarregadas = mlp_carregado.predict(amostras_teste)

assert np.array_equal(predicoes_originais, predicoes_recarregadas)

print("Validação concluída: o modelo serializado reproduz as mesmas previsões.")
print(f"Exemplo de previsões: {predicoes_recarregadas.tolist()}")

Validação concluída: o modelo serializado reproduz as mesmas previsões.
Exemplo de previsões: [7, 3, 1, 1, 2]
